# Independent BiLSTM Experiment — Soil Moisture Forecasting (TxSON33 dataset)

Goal: build a small, understandable **bidirectional LSTM** forecasting pipeline end to end (preprocessing →
feature engineering → normalization → windowing → BiLSTM → training → evaluation) on the TxSON33 modeling
dataset, and compare it against simple naive baselines (last value, moving average). This notebook replaces
the earlier 1D CNN version (`independent_cnn_experiment_txson33.ipynb`); everything CNN-specific
(convolutions, kernel size, pooling) has been removed.

Deliberately kept simple: one plain BiLSTM, **no attention, no extra engineered features**.

## Plan summary

- **Dataset:** `TxSON33_modeling_complete.parquet` (in this folder) — 33 stations, hourly, 2014-09 through
  2026-02, with every measurement carrying a matching `*_Origin` column (`observed` / `pipeline_imputed` /
  `modeling_completion`). The dataset's own README is explicit that `modeling_completion` values are synthetic
  and "should not normally be used as validation or test ground truth" — so station/year selection below is
  driven by `*_Origin` composition, not just raw row counts.
- **Station / year:** `data_quality_summary.csv` was checked across all 33 stations for `SWC_20` and `Ppt`
  observed-origin percentage. Several stations hit 100% observed for `SWC_20`, but most either cover too short a
  span or don't line up with a full calendar year. `FD11` has the longest coverage of the 100%-`SWC_20`-observed
  stations (2014-11 to 2025-07). Calendar year **2018** at `FD11` was verified directly against the parquet:
  exactly 8,760 hourly rows (no gaps, no duplicate timestamps), 100% `observed` origin for both `SWC_20` and
  `Ppt`.
- **Target:** `SWC_20` (soil water content at 20cm depth).
- **Input features:** hyperparameters are tuned with `SWC_20` + `Ppt_log`; the final input set is then chosen
  by greedy forward feature selection (Section 11) from all candidate features.
- **Lookback window:** the last **24 hours** of every input feature.
- **Prediction:** a **single value** — `SWC_20` at `h` hours after the window's last hour — for each horizon
  `h` in **24h, 48h, 72h, 168h (1 week)**. One separate model is trained per horizon.
- **Offset semantics:** input window covers hours `[t, t+23]`; the target for horizon `h` is the value at
  `t + 23 + h`. Verified with real timestamps further down.
- **Naive baselines:** *last value* (persistence) and *24h moving average*, evaluated on exactly the same test
  targets as the BiLSTM. A learned model is only useful if it beats these.
- **Split:** chronological **70% train / 10% early-stopping validation / 20% final test**. The test set is
  used exactly once, for the final comparison.
- **Hyperparameter tuning:** 5-fold `TimeSeriesSplit` cross-validation **inside the 70% training data only**.
  Starting from `hidden_size=64, num_layers=2, batch_size=16, lr=0.005`, `hidden_size` and then `num_layers`
  are tuned one at a time (15-epoch runs, 24h horizon); each value is scored by its mean validation MSE across
  the 5 folds, and the lowest mean wins.
- **Final training:** Adam, MSE loss, up to **30 epochs** with early stopping (**patience 5**) on the 10%
  validation block, one model per horizon.
- **Evaluation metrics:** MSE, MAE, SMAPE, RSE, CORR, plus MAPE for reference.
- **Forward feature selection:** greedy, at the 24h horizon, scored by the same 5-fold CV mean validation MSE.
  Step 1 tries every candidate alone; each later step adds the remaining feature that lowers CV MSE the most,
  until all features are used. The step with the lowest CV MSE gives the selected feature set.

In [ ]:
# --- Setup and reproducibility ---
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

PARQUET_PATH = "TxSON33_modeling_complete.parquet"
STATION = "FD11"
YEAR = 2018
TARGET_COL = "SWC_20"
FEATURE_COLS = ["SWC_20", "Ppt_log"]   # inputs used for hyperparameter tuning (Section 10)
CANDIDATE_FEATURES = ["SWC_20", "Ppt_log", "SWC_5", "SWC_10", "SWC_50", "T_5", "T_10", "T_20", "T_50",
                      "Tair", "RH", "Srad", "Wx", "Wy"]   # pool for forward feature selection (Section 11)
WINDOW_SIZE = 24                       # hours of past data used as input
HORIZONS = [24, 48, 72, 168]           # hours after the window's last hour; one value predicted per horizon

EPOCHS = 30            # max epochs for every main training/testing run
PATIENCE = 5           # early stopping: stop after 5 epochs without validation improvement
TUNING_EPOCHS = 15     # shorter runs used only during cross-validated tuning (Section 10)
N_FOLDS = 5            # TimeSeriesSplit folds inside the training data
TUNING_HORIZON = 24    # horizon the hyperparameters are tuned on (change to tune for a different horizon)

# Starting LSTM configuration; hidden_size and num_layers are tuned by cross-validation in Section 10
START_CONFIG = {"hidden_size": 64, "num_layers": 2, "batch_size": 16, "lr": 0.005}

# Hyperparameter sets for the final models (Section 12). Run A = result of the Section 10 CV tuning.
RUNS = {
    "Run A": {"hidden_size": 128, "num_layers": 1, "batch_size": 16, "lr": 0.005},
    # "Run B": {"hidden_size": 64, "num_layers": 2, "batch_size": 16, "lr": 0.005},
}
SELECTION_RUN = "Run A"   # hyperparameters used during forward feature selection (Section 11)

## 1. Load and inspect the data

Loading the parquet file, filtering to one station and one calendar year, and checking exactly what
that slice contains — including its `*_Origin` composition — before doing anything to it.

In [ ]:
raw = pd.read_parquet(PARQUET_PATH)
raw["Timestamp"] = pd.to_datetime(raw["Timestamp"])

df = raw[raw["Station"] == STATION].copy()
df = df[(df["Timestamp"] >= f"{YEAR}-01-01") & (df["Timestamp"] < f"{YEAR + 1}-01-01")]
df = df.sort_values("Timestamp").set_index("Timestamp")

print("Shape:", df.shape)
print("Date range:", df.index.min(), "to", df.index.max())
print("Duplicate timestamps:", df.index.duplicated().sum())
print("Hourly and gap-free:", (df.index.to_series().diff().dropna() == pd.Timedelta("1h")).all())
print()
print("Missing values in the columns used below:")
print(df[["SWC_20", "Ppt", "Wind speed", "Wind direction"]].isna().sum())
print()
print("Origin composition for the target and the precipitation feature:")
print(df["SWC_20_Origin"].value_counts())
print(df["Ppt_Origin"].value_counts())
print()
print("SWC_20 basic stats:")
print(df["SWC_20"].describe())


**Confirms the plan above:** zero missing values, no duplicate/gap timestamps, and both `SWC_20`
and `Ppt` are 100% `observed` for this station-year — none of this slice's target values are
`pipeline_imputed` or `modeling_completion`, so evaluation later is against real sensor readings.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(df.index, df["SWC_20"], linewidth=0.7)
axes[0].set_ylabel("SWC_20")
axes[0].set_title(f"{STATION}, {YEAR}: soil moisture (target) and precipitation")
axes[1].plot(df.index, df["Ppt"], linewidth=0.7, color="tab:orange")
axes[1].set_ylabel("Ppt")
axes[1].set_xlabel("Date")
plt.tight_layout()
plt.show()


`SWC_20` shows a slow-drydown-with-sharp-rises pattern — soil moisture creeps down between rain events and
jumps up when it rains. The drydown depends on how wet the soil has been over the preceding hours, and the rises
line up with precipitation. That is the kind of ordered, history-dependent signal an LSTM's memory (cell state)
is designed to carry, and it is why precipitation is included as an input from the start.

## 2. Preprocessing and feature engineering

Reproducing the relevant ideas from the project's feature-engineering conventions, written out plainly
rather than imported, keeping only what's actually used below.

### Precipitation

`log1p(Ppt)` compresses the long right tail (mostly zero, occasional storms) into a more model-friendly
range. `log1p` is undefined for values below -1, and a "did it rain" flag should never fire on sensor
noise, so negative readings are clipped to zero first — as a defensive step, not because this
particular station-year needs it (checked explicitly below).

In [ ]:
n_negative = int((df["Ppt"] < 0).sum())
print(f"Negative Ppt readings found: {n_negative} out of {len(df)} ({n_negative/len(df)*100:.2f}%)")

df["Ppt"] = df["Ppt"].fillna(0.0).clip(lower=0.0)   # defensive; no-op for this station-year
df["Ppt_log"] = np.log1p(df["Ppt"])


### Wind

`Wind speed` and `Wind direction` are stored with spaces in their column names in this dataset (not
`Windspeed`/`Winddirection` as in the older CSVs), so bracket access is needed:
`df["Wind speed"]` rather than `df.Windspeed`. Direction is also circular — 359° and 1° are nearly the
same direction, but as raw numbers they're almost as far apart as possible — so speed+direction is
converted into a 2D vector `(Wx, Wy)` via trigonometry. This also handles calm wind correctly: when
`Wind speed` is near 0, both `Wx` and `Wy` shrink toward 0 regardless of direction, since a motionless
reading carries no directional information. Wind isn't used as a model input in the experiments below —
it's included here for reference and because the column-naming difference is worth knowing about.

In [ ]:
wv = df["Wind speed"].fillna(0.0).clip(lower=0.0)
wd_rad = df["Wind direction"] * np.pi / 180.0
df["Wx"] = wv * np.cos(wd_rad)
df["Wy"] = wv * np.sin(wd_rad)

df[["Wind speed", "Wind direction", "Wx", "Wy"]].head(3)


### Missing data

Confirmed above: zero missing values and 100% `observed` origin for `SWC_20`/`Ppt` in this
station-year, so no imputation logic is needed here — worth stating explicitly rather than silently
assuming it always holds, since large parts of this dataset (per the README) do rely on
`pipeline_imputed` or `modeling_completion` values.

## 3. Chronological train / validation / test split (70 / 10 / 20)

Chronological, not random: training data must come entirely before validation data, which must come entirely
before test data, so the model is never evaluated on something that (even indirectly) informed its training.

- **Train (70%)** — used to fit the model. Hyperparameter cross-validation (Section 10) happens *inside* this
  block only.
- **Validation (10%)** — used only for early stopping of the final models (picks the best epoch).
- **Test (20%)** — touched once, at the very end, to compare the BiLSTM against the naive baselines.

In [ ]:
n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.10)

train_df = df.iloc[:n_train]
val_df = df.iloc[n_train:n_train + n_val]
test_df = df.iloc[n_train + n_val:]

print(f"train: {len(train_df):5d} rows  {train_df.index.min()} -> {train_df.index.max()}")
print(f"val:   {len(val_df):5d} rows  {val_df.index.min()} -> {val_df.index.max()}")
print(f"test:  {len(test_df):5d} rows  {test_df.index.min()} -> {test_df.index.max()}")
assert train_df.index.max() < val_df.index.min() < val_df.index.max()
assert val_df.index.max() < test_df.index.min()

print()
print("Test-period SWC_20 origin composition (must be observed ground truth, not synthetic):")
print(test_df["SWC_20_Origin"].value_counts())

## 4. Normalization

Using `MinMaxScaler`, fit on the training split only, then reused (never refit) to transform
validation and test data.

Fitting the scaler on validation or test data would leak information about their distribution into
preprocessing the model is supposed to be blind to, and would make validation/test performance look
artificially well-behaved. Fitting on train-only means validation/test values can legitimately fall
outside `[0, 1]` if that period is wetter or drier than training — expected, not a bug.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(train_df[CANDIDATE_FEATURES])   # fit on TRAIN ONLY; per-column, so any feature subset is scaled the same

def scale(split_df, cols=CANDIDATE_FEATURES, fitted_scaler=scaler):
    arr = fitted_scaler.transform(split_df[cols]).astype(np.float32)
    return pd.DataFrame(arr, columns=cols, index=split_df.index)

train_scaled = scale(train_df)
val_scaled = scale(val_df)
test_scaled = scale(test_df)

# train should be exactly [0, 1]; val/test may fall outside [0, 1] (proves no refit happened)
pd.DataFrame({"train_min": train_scaled.min(), "train_max": train_scaled.max(),
              "val_min": val_scaled.min(), "val_max": val_scaled.max(),
              "test_min": test_scaled.min(), "test_max": test_scaled.max()}).round(3)

## 5. Building X/Y windows

The core idea: slide a fixed-size 24-hour window across the hourly data; each window position becomes
one training example `X`, and its label `y` is the soil-moisture value some fixed number of hours
(`horizon`) after the window's last hour.

Written as an explicit loop rather than a vectorized one-liner, so every step is visible.

In [ ]:
def make_windows(data_df, feature_cols, target_col, window_size, horizon):
    """
    data_df: chronologically ordered, already-scaled dataframe.
    window_size: number of past hours used as input (X).
    horizon: how many hours AFTER the window's last hour the target sits.

    Returns X (num_samples, window_size, num_features), y (num_samples,), and the row-index
    (within data_df) of each window's last input hour / its target hour, for verification.
    """
    values = data_df[feature_cols].values
    targets = data_df[target_col].values
    n = len(data_df)

    X, y, end_idx, target_idx = [], [], [], []
    for start in range(0, n - window_size - horizon + 1):
        end = start + window_size          # first index AFTER the window (exclusive)
        tgt = end - 1 + horizon            # horizon hours after the window's LAST hour (end - 1)
        X.append(values[start:end])
        y.append(targets[tgt])
        end_idx.append(end - 1)
        target_idx.append(tgt)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), end_idx, target_idx


In [ ]:
# Worked example with real timestamps, horizon = 24h
X_demo, y_demo, end_idx_demo, target_idx_demo = make_windows(
    train_scaled, FEATURE_COLS, TARGET_COL, WINDOW_SIZE, horizon=24
)
i = 0
print("First input window's timestamps:")
print("  first input hour: ", train_df.index[0])
print("  last input hour:  ", train_df.index[end_idx_demo[i]])
print("  target hour:      ", train_df.index[target_idx_demo[i]])
gap_hours = (train_df.index[target_idx_demo[i]] - train_df.index[end_idx_demo[i]]).total_seconds() / 3600
print(f"  hours between window end and target: {gap_hours} (should equal the requested horizon, 24)")
print()
print("Resulting shapes: X =", X_demo.shape, " y =", y_demo.shape)
print(f"X dims: (num_samples, window_size={WINDOW_SIZE}, num_features={len(FEATURE_COLS)});  y dims: (num_samples,)")

## 6. A simple bidirectional LSTM, built from scratch

The structure follows the GeeksforGeeks PyTorch LSTM tutorial
([Long Short-Term Memory Networks using PyTorch](https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-networks-using-pytorch/)):
an `nn.LSTM` with `batch_first=True` reads the input sequence, and a single `nn.Linear` layer maps the LSTM's
final summary of the sequence to one predicted value. Two changes from that tutorial: the LSTM is
**bidirectional**, and the final summary is taken from `h_n` rather than `out[:, -1, :]` (explained below).

**Structure:**
```
input        (batch, 24 hours, F=2 features)       <- SWC_20, Ppt_log; batch_first=True, so no permute needed
nn.LSTM(input_size=F, hidden_size=H, num_layers=L, batch_first=True, bidirectional=True)
                                          <- starting point H=64, L=2; both tuned by cross-validation
  forward direction  reads hour 1 -> hour 24  -> final hidden state h_fwd  (batch, H)
  backward direction reads hour 24 -> hour 1  -> final hidden state h_bwd  (batch, H)
concat [h_fwd, h_bwd]                    (batch, 2H)
Linear(2H -> 1)                           <- single predicted SWC_20 value (scaled)
```

### What each piece does

- **LSTM cell (gates + cell state):** at every hour the LSTM updates two vectors: the *cell state* (long-term
  memory) and the *hidden state* (what it outputs at that step). Three learned gates control this: the
  **forget gate** decides how much of the old memory to keep, the **input gate** decides how much of the new
  hour's information to write, and the **output gate** decides how much of the memory to expose as the hidden
  state. For soil moisture this lets the model carry "it rained 10 hours ago" or "the soil has been drying
  steadily" forward through the window, in order — unlike the old CNN, whose average pooling threw away *when*
  something happened inside the window.
- **`input_size = F`:** number of features per hour (2: `SWC_20`, `Ppt_log`). Equivalent to the tutorial's
  `input_dim`.
- **`batch_first=True`:** tensors are `(batch, time, features)`, which is what `make_windows` already produces.
  (The CNN needed a permute to `(batch, channels, time)`; the LSTM does not.)
- **Initial hidden/cell states:** the tutorial creates `h0`/`c0` as zero tensors explicitly. PyTorch uses zeros
  automatically when none are passed, so the model below omits them — same behavior, less code. Each window
  starts from a blank memory.
- **`bidirectional=True`:** a second LSTM reads the same 24 hours in reverse. This is **not leakage**: at
  prediction time the whole 24-hour input window has already been observed, and the backward direction only
  reads hours *inside* that window — never the target hour, which is `h` hours after the window ends. The two
  directions give two summaries: one that ends on the most recent hour (the forward pass, usually the most
  informative for forecasting) and one that ends on the oldest hour.
- **Why `h_n` instead of the tutorial's `out[:, -1, :]`:** for a one-directional LSTM those are identical. For a
  bidirectional LSTM they are not: `out[:, -1, :]` holds the forward state after all 24 hours **and the backward
  state after only one hour** (the backward pass *starts* at hour 24). `h_n` holds the final state of each
  direction after it has read all 24 hours, so `h_n[-2]` (forward) and `h_n[-1]` (backward) are the correct
  summaries to concatenate.
- **`num_layers = L`:** with `L > 1` the LSTMs are stacked: the second layer reads the full hour-by-hour output
  sequence of the first layer, which lets it learn patterns built on top of the first layer's patterns. `h_n[-2]`
  and `h_n[-1]` always refer to the *last* layer.
- **`Linear(2H -> 1)`:** like the tutorial's `fc` layer: it combines the 2H-number summary into one regression
  output, with no activation function on the output (it's a regression, not a classification). The tutorial
  returns `(out, hn, cn)`; this model returns only the prediction, so the training loop stays a plain
  `pred = model(xb)`.
- **What the single output represents:** the predicted (scaled) `SWC_20` value at `window_end + horizon`.

### Hyperparameters, and why these starting values

| Setting | Starting value | Why |
|---|---|---|
| `hidden_size` | 64 per direction (tuned: 16, 32, 64, 128) | The tutorial uses 100 for a 1-feature sine wave; soil-moisture LSTM papers typically use 64–256, but with many more inputs and much more data. 64 performed best in the earlier single-split tuning run, so it is the starting point; cross-validation decides the final value. |
| `num_layers` | 2 (tuned: 1, 2, 3, 4) | The tutorial uses 1. A 2-layer stack had the lowest validation loss in the earlier single-split run; deeper stacks are tested because Kim et al. (2026) also tuned stack depth. |
| `bidirectional` | True | Requested design. LSTM vs. BiLSTM results in the literature are mixed — some soil-moisture studies find a plain LSTM equally good — so the naive baselines below are the reality check. |
| window | 24 hours | Kept from the earlier notebook. Hourly rainfall + soil-moisture LSTM studies at 20 cm have used comparable lookbacks of roughly half a day to a day. |
| optimizer / loss | Adam, MSE | Same as the tutorial. MSE matches the main evaluation metric. |
| learning rate | 0.005 (fixed) | Between Adam's default (1e-3) and the tutorial's 0.01. Not tuned in this notebook. |
| batch size | 16 (fixed) | Smaller batches gave slightly lower validation loss in the earlier single-split run. Not tuned in this notebook. |
| epochs | 30 final / 15 tuning, early stopping patience 5 | The tutorial trains a fixed 100 epochs with no validation set; here a validation block picks the best epoch, so 30 is enough while the best weights are restored. Tuning runs use 15 epochs to keep 5-fold CV affordable. |

### Related LSTM soil-moisture work (for context)

- Kim, Lee, Chung et al. (2026), *Development of long short-term memory models using rainfall and soil moisture
  to predict soil moisture dynamics*, PLOS ONE — the closest setup to this notebook: hourly in-situ soil
  moisture (20/40/60 cm) plus hourly rainfall as the only inputs, MinMax scaling, chronological
  train/val/test, grid search over window size, `num_layers`, `hidden_size`, batch size and learning rate.
  Errors grew with forecast horizon, and long horizons preferred *shorter* windows (more history acted like
  noise). [link](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0353268)
- *Enhanced global soil moisture prediction through a sampling-weighted sensitive learning strategy applied to
  various LSTM-based models* (Computers & Geosciences, 2025) — compares LSTM, BiLSTM and GRU; BiLSTM benefited
  the most from their weighting strategy.
  [link](https://www.sciencedirect.com/science/article/abs/pii/S0098300425002183)
- Frontiers in Soil Science (2025) — LSTM and BiLSTM for daily soil moisture/ET; both reached R² 0.96–0.98,
  with the plain LSTM slightly ahead. [link](https://www.frontiersin.org/journals/soil-science/articles/10.3389/fsoil.2025.1612908/full)
- *Hybrid LSTM Method for Multistep Soil Moisture Prediction Using Historical Soil Moisture and Weather Data*
  (Preprints, 2025) — multistep LSTM forecasting from past soil moisture and weather.
  [link](https://www.preprints.org/manuscript/202506.2180)

In [ ]:
class SimpleBiLSTM(nn.Module):
    def __init__(self, num_features, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(2 * hidden_size, 1)           # 2x: forward + backward summaries concatenated

    def forward(self, x):
        # x: (batch, 24, num_features)
        _, (h_n, _) = self.lstm(x)          # h_n: (num_layers * 2, batch, hidden_size); h0/c0 default to zeros
        h_forward = h_n[-2]                 # last layer, forward direction  (read hour 1 -> 24)
        h_backward = h_n[-1]                # last layer, backward direction (read hour 24 -> 1)
        h = torch.cat([h_forward, h_backward], dim=1)   # -> (batch, 2 * hidden_size)
        return self.fc(h)                   # -> (batch, 1)


_probe = SimpleBiLSTM(num_features=len(FEATURE_COLS), hidden_size=START_CONFIG["hidden_size"],
                      num_layers=START_CONFIG["num_layers"])
n_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f"Trainable parameters (F={len(FEATURE_COLS)}, starting config {START_CONFIG}): {n_params}")

# Shape check, and confirm h_n[-2] is the forward state at the LAST hour while h_n[-1] is the backward
# state at the FIRST hour -- i.e. both directions have read the full window.
_x = torch.randn(4, WINDOW_SIZE, len(FEATURE_COLS))
_out, (_h_n, _) = _probe.lstm(_x)
H = _probe.lstm.hidden_size
assert _probe(_x).shape == (4, 1)
assert torch.allclose(_h_n[-2], _out[:, -1, :H])   # forward final state == forward output at hour 24
assert torch.allclose(_h_n[-1], _out[:, 0, H:])    # backward final state == backward output at hour 1
print("Output shape (4, 1) and h_n direction checks passed.")

## 7. Evaluation metrics, verified independently

Reproducing the same 5-metric set the project's evaluator reports (MSE, MAE, SMAPE, RSE, CORR), written
from the mathematical definitions rather than copied, and sanity-checked against known values before
trusting them on real data. MAPE is included too, with an explicit caveat.

MAPE divides by the actual value, so it blows up when the target is near zero. `SWC_20` here ranges
roughly 0.18-0.5 (never near zero), so MAPE is less dangerous for this particular target than it would
be for something like precipitation — but it's still the least robust of these metrics in general, so
SMAPE is reported as the primary percentage-error metric, with MAPE kept alongside only for reference.

In [ ]:
def compute_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64).flatten()
    y_pred = np.asarray(y_pred, dtype=np.float64).flatten()

    mse = np.mean((y_true - y_pred) ** 2)
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps))) * 100
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100
    rse = np.sum((y_pred - y_true) ** 2) / (np.sum((y_true - y_true.mean()) ** 2) + eps)
    corr = np.corrcoef(y_true, y_pred)[0, 1] if np.std(y_pred) > 1e-12 else float("nan")

    return {"MSE": mse, "MAE": mae, "MAPE": mape, "SMAPE": smape, "RSE": rse, "CORR": corr}


# Sanity checks against known cases
perfect = compute_metrics([1, 2, 3], [1, 2, 3])
assert perfect["MSE"] == 0 and perfect["MAE"] == 0 and abs(perfect["CORR"] - 1.0) < 1e-9
mape_check = compute_metrics([1.0], [0.0])
assert abs(mape_check["MAPE"] - 100.0) < 1e-6, mape_check["MAPE"]
print("Metric sanity checks passed: perfect prediction -> zero error, CORR=1; true=1,pred=0 -> MAPE=100%.")


## 8. Training loop

A plain, manual PyTorch loop: no trainer framework, no callback system — just the steps, visible.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE, lr=1e-3, verbose=False):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    train_losses, val_losses = [], []
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()                          # training mode
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
        train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()                           # evaluation mode
        val_running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                val_running += criterion(pred, yb).item() * xb.size(0)
        val_loss = val_running / len(val_loader.dataset)
        val_losses.append(val_loss)

        if verbose:
            print(f"  epoch {epoch+1:2d}: train_loss={train_loss:.6f} val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - 1e-7:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)      # restore the best-validation epoch

    return model, train_losses, val_losses

In [ ]:
def run_experiment(horizon, feature_cols=FEATURE_COLS, build_model=None, splits=None, scaler_obj=scaler,
                   batch_size=32, lr=1e-3, epochs=EPOCHS, patience=PATIENCE, seed=SEED, verbose=False):
    """
    Trains one model for one horizon and evaluates it on the test split.

    feature_cols: input columns (must be columns of the scaled splits and of scaler_obj).
    build_model:  function num_features -> nn.Module; defaults to SimpleBiLSTM with its default settings.
    splits:       (train_scaled, val_scaled, test_scaled); defaults to the 70/10/20 splits built above.
    """
    t0 = time.time()
    if build_model is None:
        build_model = lambda nf: SimpleBiLSTM(nf)
    train_s, val_s, test_s = splits if splits is not None else (train_scaled, val_scaled, test_scaled)

    Xtr, ytr, _, _ = make_windows(train_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
    Xv, yv, _, _ = make_windows(val_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
    Xte, yte, _, _ = make_windows(test_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)

    g = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)),
                               batch_size=batch_size, shuffle=True, generator=g)
    val_loader = DataLoader(TensorDataset(torch.tensor(Xv), torch.tensor(yv).unsqueeze(1)), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte).unsqueeze(1)), batch_size=batch_size)

    torch.manual_seed(seed)
    model = build_model(len(feature_cols))
    model, train_losses, val_losses = train_model(model, train_loader, val_loader, epochs=epochs,
                                                    patience=patience, lr=lr, verbose=verbose)

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
            trues.append(yb.numpy())
    preds = np.concatenate(preds).flatten()
    trues = np.concatenate(trues).flatten()

    # back to the original SWC_20 scale for interpretable metrics
    target_idx = list(scaler_obj.feature_names_in_).index(TARGET_COL)
    col_min, col_max = scaler_obj.data_min_[target_idx], scaler_obj.data_max_[target_idx]
    preds_orig = preds * (col_max - col_min) + col_min
    trues_orig = trues * (col_max - col_min) + col_min

    metrics = compute_metrics(trues_orig, preds_orig)
    metrics["best_val_loss"] = min(val_losses)   # scaled MSE on the validation split -- used for choosing settings
    metrics["runtime_sec"] = time.time() - t0
    metrics["epochs_run"] = len(train_losses)

    return {"metrics": metrics, "y_true": trues_orig, "y_pred": preds_orig,
            "train_losses": train_losses, "val_losses": val_losses}

## 9. Naive baselines: last value and moving average

Two models with nothing to learn, evaluated on **exactly the same test targets** as the BiLSTM (same windows,
same horizons). Soil moisture is very persistent hour to hour, so these are hard to beat at short horizons — a
BiLSTM that doesn't beat them isn't adding anything yet.

- **Last value (persistence):** predict that `SWC_20` at `window_end + h` equals `SWC_20` at the window's last
  hour.
- **Moving average (24h):** predict the mean `SWC_20` over the 24-hour input window. Smoother than last value,
  but lags behind trends.

Computed directly from the **unscaled** test split, so no inverse scaling is involved.

In [ ]:
def naive_baselines(data_df, horizon, ma_window=WINDOW_SIZE):
    """Returns the true targets and naive predictions for every window, in original SWC_20 units."""
    X, y, _, _ = make_windows(data_df, [TARGET_COL], TARGET_COL, WINDOW_SIZE, horizon)
    swc = X[:, :, 0]
    return y, {
        "Last value": swc[:, -1],
        f"Moving average ({ma_window}h)": swc[:, -ma_window:].mean(axis=1),
    }


naive_results = {}   # naive_results[h][name] = {"metrics": ..., "y_pred": ...}
for h in HORIZONS:
    y_true_naive, naive_preds = naive_baselines(test_df, h)
    naive_results[h] = {name: {"metrics": compute_metrics(y_true_naive, pred), "y_pred": pred, "y_true": y_true_naive}
                        for name, pred in naive_preds.items()}

NAIVE_NAMES = list(naive_results[HORIZONS[0]].keys())
naive_table = pd.concat(
    {name: pd.DataFrame({h: naive_results[h][name]["metrics"] for h in HORIZONS}).T for name in NAIVE_NAMES},
    names=["baseline", "horizon_hours"])
naive_table

## 10. Tuning `hidden_size` and `num_layers` with time-series cross-validation

All tuning happens **inside the 70% training block** — the 10% validation and 20% test blocks are never seen
here, so the final test numbers stay an honest estimate.

**How the folds work (`sklearn.model_selection.TimeSeriesSplit`, 5 folds):** the training block is cut into 6
consecutive chunks. Fold 1 trains on chunk 1 and validates on chunk 2; fold 2 trains on chunks 1–2 and validates
on chunk 3; … fold 5 trains on chunks 1–5 and validates on chunk 6. Each fold always trains on the past and
validates on the period right after it, so no future information leaks into training (plain `KFold` would
sometimes train on later months and validate on earlier ones).

**Inside each fold:**
- a new `MinMaxScaler` is fit on that fold's training part only, then applied to its validation part,
- windows are built separately for each part, so no input window or target crosses the fold boundary,
- the model trains for up to **15 epochs** (`TUNING_EPOCHS`) with patience 5, and the fold's score is its best
  validation MSE, converted back to original `SWC_20` units so scores from different folds are comparable.

**Choosing a value:** the score of a setting is the **mean validation MSE across the 5 folds**; the lowest mean
wins. Settings are tuned one at a time, starting from `START_CONFIG`, at the `TUNING_HORIZON` (24h):

| Step | Setting being tested | Values | Held fixed |
|---|---|---|---|
| 10.1 | `hidden_size` | 16, 32, 64, 128 | `num_layers=2`, `batch_size=16`, `lr=0.005` |
| 10.2 | `num_layers` | 1, 2, 3, 4 | best `hidden_size` from 10.1, `batch_size=16`, `lr=0.005` |

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=N_FOLDS)
cv_folds = list(tscv.split(train_df))

for k, (tr_idx, va_idx) in enumerate(cv_folds, start=1):
    tr, va = train_df.iloc[tr_idx], train_df.iloc[va_idx]
    assert tr.index.max() < va.index.min()   # always train on the past, validate on the future
    print(f"fold {k}: train {len(tr):5d} rows {tr.index.min():%Y-%m-%d} -> {tr.index.max():%Y-%m-%d} | "
          f"val {len(va):5d} rows {va.index.min():%Y-%m-%d} -> {va.index.max():%Y-%m-%d}")

In [ ]:
def build_bilstm(config):
    return lambda nf: SimpleBiLSTM(nf, hidden_size=config["hidden_size"], num_layers=config["num_layers"])


def cv_score(config, horizon=TUNING_HORIZON, feature_cols=FEATURE_COLS, epochs=TUNING_EPOCHS):
    """Mean (and per-fold) best validation MSE, in original SWC_20 units, across the TimeSeriesSplit folds."""
    scale_cols = list(dict.fromkeys(list(feature_cols) + [TARGET_COL]))   # inputs + target, no duplicates
    target_pos = scale_cols.index(TARGET_COL)
    fold_mses = []
    for tr_idx, va_idx in cv_folds:
        fold_train, fold_val = train_df.iloc[tr_idx], train_df.iloc[va_idx]

        fold_scaler = MinMaxScaler().fit(fold_train[scale_cols])       # fit on this fold's training part only
        tr_s = scale(fold_train, scale_cols, fold_scaler)
        va_s = scale(fold_val, scale_cols, fold_scaler)

        Xtr, ytr, _, _ = make_windows(tr_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
        Xv, yv, _, _ = make_windows(va_s, feature_cols, TARGET_COL, WINDOW_SIZE, horizon)
        g = torch.Generator().manual_seed(SEED)
        train_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)),
                                   batch_size=config["batch_size"], shuffle=True, generator=g)
        val_loader = DataLoader(TensorDataset(torch.tensor(Xv), torch.tensor(yv).unsqueeze(1)),
                                 batch_size=config["batch_size"])

        torch.manual_seed(SEED)
        model = build_bilstm(config)(len(feature_cols))
        _, _, val_losses = train_model(model, train_loader, val_loader, epochs=epochs, patience=PATIENCE,
                                        lr=config["lr"])

        # scaled MSE -> original units: (x_scaled error) * (max - min)  =>  MSE * (max - min)^2
        target_range = fold_scaler.data_max_[target_pos] - fold_scaler.data_min_[target_pos]
        fold_mses.append(min(val_losses) * target_range ** 2)
    return float(np.mean(fold_mses)), fold_mses


tuned_config = dict(START_CONFIG)   # updated after each step
_cv_cache = {}                      # avoids re-running a config that was already scored

def tune_setting_cv(setting_name, values):
    """Varies ONE setting, holding every other value of tuned_config fixed; picks the lowest mean CV MSE."""
    fixed = {k: v for k, v in tuned_config.items() if k != setting_name}
    print(f"--- Testing {setting_name} = {values}  (fixed: {fixed}, horizon={TUNING_HORIZON}h, "
          f"{N_FOLDS} folds, {TUNING_EPOCHS} epochs) ---")
    rows = {}
    for v in values:
        config = {**tuned_config, setting_name: v}
        key = tuple(sorted(config.items()))
        t0 = time.time()
        if key not in _cv_cache:
            _cv_cache[key] = cv_score(config)
        mean_mse, fold_mses = _cv_cache[key]
        rows[v] = {"mean_val_MSE": mean_mse, "std_val_MSE": np.std(fold_mses),
                   **{f"fold{k}_MSE": m for k, m in enumerate(fold_mses, start=1)}}
        print(f"  {setting_name}={v}: mean val MSE={mean_mse:.7f}  ({time.time() - t0:.0f}s)")
    table = pd.DataFrame(rows).T
    table.index.name = setting_name
    best = type(values[0])(table["mean_val_MSE"].idxmin())
    tuned_config[setting_name] = best
    print(f"  -> best {setting_name}: {best}")
    return table

## 11. Greedy forward feature selection (24h horizon)

In [ ]:
sel_config = RUNS[SELECTION_RUN]
print(f"Selection config ({SELECTION_RUN}): {sel_config}, horizon={TUNING_HORIZON}h, {N_FOLDS} folds, "
      f"{TUNING_EPOCHS} epochs")

selected, remaining = [], list(CANDIDATE_FEATURES)
step_rows = []          # best feature set at each step
candidate_scores = {}   # candidate_scores[step][feature] = mean CV MSE when that feature is added

step = 0
while remaining:
    step += 1
    print(f"\n--- Step {step}: adding one of {len(remaining)} features to {selected} ---")
    scores = {}
    for f in remaining:
        t0 = time.time()
        mean_mse, fold_mses = cv_score(sel_config, feature_cols=selected + [f])
        scores[f] = (mean_mse, float(np.std(fold_mses)))
        print(f"  + {f:8s}: mean val MSE={mean_mse:.7f}  ({time.time() - t0:.0f}s)")
    best_f = min(scores, key=lambda f: scores[f][0])
    candidate_scores[step] = {f: s[0] for f, s in scores.items()}
    selected = selected + [best_f]
    remaining.remove(best_f)

    # test error of this feature set, for reference only -- NOT used to choose features
    test_r = run_experiment(TUNING_HORIZON, feature_cols=selected, build_model=build_bilstm(sel_config),
                            batch_size=sel_config["batch_size"], lr=sel_config["lr"])
    step_rows.append({"n_features": step, "added": best_f, "cv_mean_val_MSE": scores[best_f][0],
                      "cv_std_val_MSE": scores[best_f][1], "test_MSE": test_r["metrics"]["MSE"],
                      "features": list(selected)})
    print(f"  -> added {best_f}: CV MSE={scores[best_f][0]:.7f}, test MSE (reference)={test_r['metrics']['MSE']:.7f}")

selection_table = pd.DataFrame(step_rows).set_index("n_features")
cv_curve = selection_table["cv_mean_val_MSE"]
first_dip = next((s for s in cv_curve.index[1:] if cv_curve[s] > cv_curve[s - 1]), None)
best_step = int(cv_curve.idxmin())
SELECTED_FEATURES = selection_table.loc[best_step, "features"]

print(f"\nCV error first got worse when going to {first_dip} features." if first_dip else
      "\nCV error never got worse while adding features.")
print(f"Lowest CV MSE at {best_step} features -> SELECTED_FEATURES = {SELECTED_FEATURES}")
selection_table

In [ ]:
# Mean CV MSE of every candidate tried at every step (blank = already selected)
candidate_table = pd.DataFrame(candidate_scores).T[CANDIDATE_FEATURES]
candidate_table.index.name = "step"
candidate_table.style.format("{:.7f}", na_rep="").highlight_min(axis=1, color="lightgreen")

In [ ]:
x = selection_table.index
plt.figure(figsize=(11, 4.5))
plt.errorbar(x, selection_table["cv_mean_val_MSE"], yerr=selection_table["cv_std_val_MSE"], marker="o",
             capsize=3, label=f"CV mean val MSE ({N_FOLDS} folds, ± std)")
plt.plot(x, selection_table["test_MSE"], marker="s", linestyle="--", label="Test MSE (reference only)")
plt.axvline(best_step, color="gray", linestyle=":", label=f"selected: {best_step} features")
plt.xticks(x, [f"{n}\n+{f}" for n, f in zip(x, selection_table["added"])], fontsize=8)
plt.xlabel("Number of features (feature added at that step)")
plt.ylabel("MSE (SWC_20 units)")
plt.title(f"Greedy forward feature selection, {TUNING_HORIZON}h horizon ({SELECTION_RUN})")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Final BiLSTM at every horizon (selected features)

In [ ]:
print("Selected features:", SELECTED_FEATURES)
final_results = {run_name: {} for run_name in RUNS}
for run_name, config in RUNS.items():
    for h in HORIZONS:
        print(f"Training {run_name} {config}, horizon={h}h ...")
        r = run_experiment(h, feature_cols=SELECTED_FEATURES, build_model=build_bilstm(config),
                           batch_size=config["batch_size"], lr=config["lr"])
        final_results[run_name][h] = r
        print(f"  done in {r['metrics']['runtime_sec']:.1f}s, {r['metrics']['epochs_run']} epochs")

        # the BiLSTM and the naive baselines must be scored on the identical test targets
        assert np.allclose(r["y_true"], naive_results[h][NAIVE_NAMES[0]]["y_true"], atol=1e-5)

## 13. Results by horizon

In [ ]:
from IPython.display import display

METRIC_COLS = ["MSE", "MAE", "MAPE", "SMAPE", "RSE", "CORR"]

def show_horizon(h):
    table = pd.DataFrame({**{f"BiLSTM ({run_name})": final_results[run_name][h]["metrics"] for run_name in RUNS},
                          **{name: naive_results[h][name]["metrics"] for name in NAIVE_NAMES}}).T[METRIC_COLS]
    table.index.name = f"{h}h horizon"
    display(table)

    _, _, _, target_idx = make_windows(test_df, [TARGET_COL], TARGET_COL, WINDOW_SIZE, h)
    target_times = test_df.index[target_idx]
    plt.figure(figsize=(11, 4))
    first_run = next(iter(RUNS))
    plt.plot(target_times, final_results[first_run][h]["y_true"], label="Actual", linewidth=1.2, color="black")
    for run_name in RUNS:
        plt.plot(target_times, final_results[run_name][h]["y_pred"], label=f"BiLSTM ({run_name})", linewidth=1)
    for name in NAIVE_NAMES:
        plt.plot(target_times, naive_results[h][name]["y_pred"], label=name, linewidth=0.8, alpha=0.7,
                 linestyle="--")
    plt.title(f"{STATION} {YEAR}: SWC_20 actual vs. predicted, {h}h horizon (test period)")
    plt.ylabel("SWC_20")
    plt.xlabel("Target time")
    plt.legend()
    plt.tight_layout()
    plt.show()

### 13.1 Horizon: 24 hours

In [ ]:
show_horizon(24)

### 13.2 Horizon: 48 hours

In [ ]:
show_horizon(48)

### 13.3 Horizon: 72 hours

In [ ]:
show_horizon(72)

### 13.4 Horizon: 168 hours (1 week)

In [ ]:
show_horizon(168)